# Meta: Baseline Evaluations

In [1]:
# Imports
import os, json, sys
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
import numpy as np
import seaborn as sns
import language_tool_python as ltp
from typing import List, Dict, Any

In [2]:
# Paths
notebook_path = os.getcwd()
project_root = os.path.dirname(os.path.abspath(notebook_path))
data_dir = os.path.join(project_root, "data")
evaluations_dir = os.path.join(data_dir, "evaluations")
responses_dir = os.path.join(data_dir, "generated")
vocab_dir = os.path.join(data_dir, "vocab")
db_path = os.path.join(data_dir, "DB.db")


In [3]:
# SQL Alchemy session
engine = create_engine(f"sqlite:///{db_path}")
session_factory = sessionmaker(bind=engine)
session = session_factory()

In [4]:
# Helper functions
def get_prompt_eval_dir(prompt_id):
    return os.path.join(evaluations_dir, f"prompt{prompt_id}")

def get_unique_ids(prompt_eval_dir):
    unique_ids = []
    for filename in os.listdir(prompt_eval_dir):
        if filename.endswith(".json"):
            unique_ids.append(filename.split(".json")[0])
    return unique_ids

def load_evaluation(prompt_id):
    prompt_dir = get_prompt_eval_dir(prompt_id)
    unique_ids = get_unique_ids(prompt_dir)
    res = []
    for unique_id in unique_ids:
        file_path = os.path.join(prompt_dir, f"{unique_id}.json")
        with open(file_path, "r", encoding="utf-8") as file:
            res.append(json.load(file))
    return res

def load_evaluations(prompt_ids):
    res_list = []
    for prompt_id in prompt_ids:
        res_list.append(load_evaluation(prompt_id))
    return res_list

In [15]:
# Ground truth
gt_id = "gt.45af4a46-d238-4251-b734-d407602299bd"
gtb_id = "gtb.45af4a46-d258-4251-b734-d407602299bd"

gt_eval_path = os.path.join(evaluations_dir, f"{gt_id}.json")
gtb_eval_path = os.path.join(evaluations_dir, f"{gtb_id}.json")

gt_path = os.path.join(responses_dir, f"{gt_id}.json")
gtb_path = os.path.join(responses_dir, f"{gtb_id}.json")

with open(gt_path, "r", encoding="utf-8") as file:
    gt_res = json.load(file)
with open(gt_path, "r", encoding="utf-8") as file:
    gtb_res = json.load(file)

In [16]:
print(gt_res.keys())

dict_keys(['model', 'prompt', 'unique_id', 'responses'])


In [18]:
def get_whitelist(vocab_dir: str):
    """Get the whitelist from the vocab directory."""
    with open(os.path.join(vocab_dir, "fp.json"), "r", encoding="utf-8") as file:
        whitelist = json.load(file)
    return whitelist

In [7]:
lang_tool = ltp.LanguageTool(language='de-De', remote_server='localhost:8010')

In [ ]:
# Language Tool Metrics
def language_tool_spelling_check(lang_tool: ltp.LanguageTool, output_report, vocab_dir):
    """function to check the output report using language tool"""
    lang_tool.enabled_rules_only = True
    lang_tool.enabled_categories = {"TYPOS"}

    typos = []
    typos_count = 0
    whitelist_count = 0
    whitelist = [] #get_whitelist(vocab_dir)

    try:
        matches = lang_tool.check(output_report)
        for match in matches:
            try:
                word = match.context[
                    match.offsetInContext : match.offsetInContext + match.errorLength
                ]
                if word in whitelist:
                    whitelist_count += 1
                    typos.append((word, match.context, False))
                else:
                    typos_count += 1
                    typos.append((word, match.context, True))
            except Exception as e:
                print(e)
    except ltp.utils.LanguageToolError as e:
        print(e)

    return typos, typos_count, whitelist_count


def language_tool_grammar_check(lang_tool: ltp.LanguageTool, output_report):
    """function to check the output report using language tool"""
    lang_tool.enabled_rules_only = True
    lang_tool.enabled_categories = {"GRAMMAR"}

    grammar = []

    try:
        matches = lang_tool.check(output_report)
        for match in matches:
            grammar.append(match.ruleId)
    except ltp.utils.LanguageToolError:
        grammar.append("LT Error")

    return grammar, len(matches)

def language_tool_other_check(lang_tool: ltp.LanguageTool, output_report):
    """function to check the output report using language tool"""
    lang_tool.enabled_rules_only = True
    lang_tool.enabled_categories = {}

    other = []

    try:
        matches = lang_tool.check(output_report)
        for match in matches:
            other.append(match.ruleId)
    except ltp.utils.LanguageToolError:
        other.append("LT Error")

    return other, len(matches)


def get_language_tool_metrics(
    responses: Dict[str, Any], lang_tool: ltp.LanguageTool, directories: List[str]
) -> Dict[str, Any]:
    """function to get the language tool metrics"""
    language_tool_metrics = {
        "counts": {
            "typos": 0,
            "whitelist": 0,
            "grammar": 0,
            "other": 0,
        },
        "typos": {},
        "grammar": {},
        "other": {},
    }

    for response in responses:
        ris_id = response["ris_id"]
        output_report = response["raw"]["response"]
        try:
            typos, typos_count, whitelist_count = language_tool_spelling_check(lang_tool, output_report, directories[0])
            language_tool_metrics["counts"]["typos"] += typos_count
            language_tool_metrics["counts"]["whitelist"] += whitelist_count
            language_tool_metrics["typos"].update({ris_id: typos})
        except Exception as e:
            print(e)
            continue
        try:
            grammar, count = language_tool_grammar_check(lang_tool, output_report)
            language_tool_metrics["counts"]["grammar"] += count
            language_tool_metrics["grammar"].update({ris_id: grammar})
        except Exception as e:
            print(e)
            continue
        try:
            other, count = language_tool_other_check(lang_tool, output_report)
            language_tool_metrics["counts"]["other"] += count
            language_tool_metrics["other"].update({ris_id: other})
        except Exception as e:
            print(e)
            continue
    return language_tool_metrics

In [22]:
matches_dict = {}
for response in gt_res["responses"]:
    matches = lang_tool.check(response["raw"]["response"])
    matches_dict.update({response["ris_id"]: matches})

In [23]:
matches_dict_b = {}
for response in gtb_res["responses"]:
    matches = lang_tool.check(response["raw"]["response"])
    matches_dict_b.update({response["ris_id"]: matches})

In [ ]:
rule_set = set()
cat_set = set()
for matches in matches_dict_b.values():
    for match in matches:
        rule_set.add(match.ruleId)
        cat_set.add(match.category)

In [54]:
print(rule_set)
print(cat_set)

{'ZAHL_IM_WORT_SPELLING_RULE', 'GERMAN_SPELLER_RULE', 'LEERZEICHEN_UND', 'ZUR_DER'}
{'TYPOS'}


In [55]:
words = set()
for matches in matches_dict.values():
    for match in matches:
        words.add(match.context[match.offsetInContext : match.offsetInContext + match.errorLength])
for matches in matches_dict_b.values():
    for match in matches:
        words.add(match.context[match.offsetInContext : match.offsetInContext + match.errorLength])

In [69]:
new_matches = []
lang_tool = ltp.LanguageTool(language='en-US', remote_server='localhost:8010')
for word in words:
    new_matches.append(lang_tool.check(word))

In [ ]:
words_list = list(words)
with open("words.json", "w", encoding="utf-8") as json_file:
    json.dump(words_list, json_file, ensure_ascii=False)

In [68]:
print(len(new_matches))
print(len(words))

1809
1809


## Whole Reports

## Befunds